# Lesson 05 Lab — Unstructured Magnitude Pruning Without Storage Myths

**Puzzle:** Why can a model contain 80% zeros while its ordinary state_dict grows?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Unstructured magnitude pruning is easy to apply and useful for studying redundancy, but PyTorch's training-time reparameterization stores the original parameter and a mask. Logical zeros, raw checkpoint bytes, compressed bytes, and physical sparse storage are four distinct quantities.


## 0. Predict before running

1. Predict the state_dict keys immediately after PyTorch pruning.
2. Predict whether the raw serialized bytes shrink after `prune.remove`.
3. Predict which representation gzip compresses most effectively.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

The experiment uses one linear module before pruning, after `l1_unstructured`, after `prune.remove`, and after gzip compression. It inspects parameter names, buffers, zero rate, forward equivalence, and serialized byte counts.

- A mask changes parameterization before it changes storage format.
- Removing the reparameterization materializes zeros but keeps a dense tensor.
- Raw and compressed checkpoint sizes answer different deployment questions.


## 2. Derive the mechanism

PyTorch pruning replaces `weight` with `weight_orig` and computes `weight_orig × weight_mask` through a pre-hook. The dense tensors still occupy dense storage, and the additional mask can make an uncompressed state_dict larger. `prune.remove` materializes the masked weight and deletes the reparameterization but does not convert it to CSR or pack nonzeros. General-purpose compression can exploit repeated zero bytes, which explains why compressed file size may fall while raw tensor storage does not.

### Mechanism at a glance

```mermaid
flowchart LR
  W["dense weight"] --> A["apply pruning"]
  A --> P["weight_orig + weight_mask"]
  P --> F["forward uses weight_orig × mask"]
  P --> R["prune.remove()"]
  R --> M["materialized dense tensor<br/>containing zeros"]
  M --> S["optional compression or<br/>explicit sparse encoding"]
```

### Walk it step by step

1. **Apply the mask.** PyTorch stores the original parameter and a mask, then computes their product through a hook.
2. **Audit logical sparsity.** Count zeros and verify forward behavior without making a storage claim.
3. **Remove the reparameterization.** Materialize the masked dense tensor and confirm state_dict keys and load behavior.
4. **Choose an actual storage format.** Compression, CSR, and backend-specific packing answer different deployment questions.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 5
LESSON_TITLE = 'Unstructured Magnitude Pruning Without Storage Myths'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260813
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | the original dense linear layer state_dict |
| Candidate | PyTorch pruning reparameterization and the materialized masked weight |
| Held constant | same weight values, pruning amount, serializer, compression level, and module shape |
| Measurements | zero rate, state_dict keys, raw bytes, gzip bytes, and forward equivalence |
| Evidence | `pytorch-gpu` |

**Experiment:** Trace an 80% magnitude mask through apply, save, remove, and compressed serialization stages.


## 5. Read the experiment code

BytesIO keeps the serialization experiment inside memory and avoids path-dependent artifacts. The notebook records keys before and after `remove`, evaluates the module around the transition, and compresses the same byte payload. It does not call the resulting file a sparse runtime format.

Do not execute until the code implements the frozen table above.


In [2]:
import torch.nn.utils.prune as prune
layer = nn.Linear(1024, 1024, bias=False, device=DEVICE)
probe = torch.randn(8, 1024, device=DEVICE)

def serialized(state):
    buffer = io.BytesIO(); torch.save(state, buffer); raw = buffer.getvalue()
    return len(raw), len(gzip.compress(raw, compresslevel=9))

dense_output = layer(probe).detach()
dense_raw, dense_gzip = serialized(layer.state_dict())
prune.l1_unstructured(layer, name="weight", amount=0.80)
hook_output = layer(probe).detach()
hook_keys = sorted(layer.state_dict().keys())
hook_raw, hook_gzip = serialized(layer.state_dict())
logical_sparsity = zero_fraction(layer.weight)
prune.remove(layer, "weight")
removed_output = layer(probe).detach()
removed_keys = sorted(layer.state_dict().keys())
removed_raw, removed_gzip = serialized(layer.state_dict())
metrics = {
    "sparsity": logical_sparsity,
    "dense_raw_bytes": dense_raw,
    "dense_gzip_bytes": dense_gzip,
    "hook_raw_bytes": hook_raw,
    "hook_gzip_bytes": hook_gzip,
    "removed_raw_bytes": removed_raw,
    "removed_gzip_bytes": removed_gzip,
    "hook_keys": hook_keys,
    "removed_keys": removed_keys,
    "remove_max_error": float((removed_output - hook_output).abs().max().item()),
    "pruning_change_rmse": tensor_metrics(dense_output, removed_output)["rmse"],
}
analysis = (
    f"The effective weight reached {logical_sparsity:.1%} sparsity. The hook checkpoint used keys "
    f"{hook_keys} and occupied {hook_raw:,} raw bytes, while `remove` restored a single key {removed_keys} "
    f"and {removed_raw:,} raw bytes. Gzip reduced the materialized payload to {removed_gzip:,} bytes. "
    f"Forward drift across `remove` was {metrics['remove_max_error']:.3e}, proving lifecycle equivalence but not sparse storage."
)


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Logical sparsity | 80.00% |
| Dense raw bytes | 4,195,945 bytes |
| Pruned-hook raw bytes | 8,390,501 bytes |
| Removed raw bytes | 4,195,945 bytes |
| Removed gzip bytes | 1,120,583 bytes |
| Remove max output drift | 0.000000 |


## 7. Interpret rather than merely print

The effective weight reached 80.0% sparsity. The hook checkpoint used keys ['weight_mask', 'weight_orig'] and occupied 8,390,501 raw bytes, while `remove` restored a single key ['weight'] and 4,195,945 raw bytes. Gzip reduced the materialized payload to 1,120,583 bytes. Forward drift across `remove` was 0.000e+00, proving lifecycle equivalence but not sparse storage.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 5,
    "title": 'Unstructured Magnitude Pruning Without Storage Myths',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Magnitude pruning creates zeros; storage compression and runtime acceleration require additional explicit representations.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 5,
  "title": "Unstructured Magnitude Pruning Without Storage Myths",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260813
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "sparsity": 0.8000001907348633,
    "dense_raw_bytes": 4195945,
    "dense_gzip_bytes": 3794967,
    "hook_raw_bytes": 8390501,
    "hook_gzip_bytes": 3961000,
    "removed_raw_bytes": 4195945,
    "removed_gzip_bytes": 1120583,
    "hook_keys": [
      "weight_mask",
      "weight_orig"
    ],
    "removed_keys": [
      "weight"
    ],
    "remove_max_error": 0.0,
    "pruning_change_rmse": 0.4095126986503601
  },
  "analysis": "The effective weight reached 80.0% sparsity. The hook checkpoint used keys ['weight_mask', 'weight_orig'] and occupied 8,390,501 raw bytes, while `remove` restored a single key ['weight'] and 4,195,945 raw bytes. Gzip reduced the mate

## 9. Make the bounded decision

> Magnitude pruning creates zeros; storage compression and runtime acceleration require additional explicit representations.

**Acceptance/rollback:** Accept the mask lifecycle only when the saved keys, zero rate, load path, and intended deployment representation are explicitly tested.

**Failure analysis:** File systems and zip serialization can introduce version-dependent overhead, so tiny tensors exaggerate metadata. Gzip size is not resident GPU memory and says nothing about kernel speed. A deployment claiming sparse storage must identify the actual sparse encoding and loader.


## 10. Extend the evidence

Convert the materialized matrix to CSR and compare metadata plus supported operations; then load every saved variant into a fresh process and verify outputs before benchmarking.

The full evidence boundary and references are in [`README.md`](README.md).
